# HCDE 530 — Week 5: Exploring `app_reviews_demo.csv`

This notebook follows the same **first-look** pattern as `week5_pandas_demo.ipynb`, using **`app_reviews_demo.csv`** — one row per app review (rating, text, device, dates, etc.).

Run cells top to bottom (**Shift + Enter**). Your working directory should be **`hcde530_week5`** so the CSV path resolves.

---

**Questions we answer:**

1. What does the dataset look like? (`head()`, `info()`)
2. What is the distribution of the most important column?
3. Filter to a meaningful subset — what’s in it?
4. Group by a category and take the average of a numeric column.
5. What are the missing values? Are any columns incomplete?

---

## Setup

Import **pandas** and load the CSV from this folder.

In [1]:
import pandas as pd
import warnings

warnings.filterwarnings("ignore")
print("pandas version:", pd.__version__)

df = pd.read_csv("app_reviews_demo.csv")

pandas version: 3.0.2


---
## 1. What does your dataset look like? `head()`, `info()`

**`head()`** shows the first rows. **`info()`** lists column names, non-null counts, and dtypes.

In [2]:
df.head()

,id,app,category,rating,review,date,helpful_votes,verified_purchase,device_type,app_version
0,1,Fieldkit,field research,1,Auto-transcription accuracy on accented speake...,2023-03-31,37,True,mobile,2.5.0
1,2,Fieldkit,field research,2,Search results are slow when the repository is...,2024-07-28,12,True,mobile,2.5.3
2,3,Lookback,user research,4,One-click export to Notion is a feature I use ...,2024-03-08,21,True,desktop,5.2.0
3,4,Dovetail,research repository,5,My whole team can comment on the same session ...,2023-12-19,38,True,NaN,2.0.0
4,5,Fieldkit,field research,5,Works offline and syncs when I get back to WiF...,2024-01-23,5,True,desktop,2.5.3


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   id                 500 non-null    int64
 1   app                500 non-null    str  
 2   category           500 non-null    str  
 3   rating             500 non-null    int64
 4   review             500 non-null    str  
 5   date               500 non-null    str  
 6   helpful_votes      500 non-null    int64
 7   verified_purchase  500 non-null    bool 
 8   device_type        437 non-null    str  
 9   app_version        389 non-null    str  
dtypes: bool(1), int64(3), str(6)
memory usage: 35.8 KB


**Readout:** Each row is one **review**: identifiers (`id`, `app`), **`category`** (type of research tool), **`rating`** (1–5), **`review`** text, **`date`**, **`helpful_votes`**, **`verified_purchase`**, **`device_type`**, **`app_version`**. Row count and dtypes come from `info()` above (expect **500** reviews in the demo file).

---
## 2. What is the distribution of your most important column?

For review data, **`rating`** is usually the key outcome: it summarizes satisfaction in one number. We use **`value_counts()`** sorted by rating so you see **how many reviews fall at each star level** (the distribution of scores).

We also show **`describe()`** on **`rating`** for mean, spread, and quartiles.

In [4]:
# Distribution of star ratings (most important numeric outcome for this dataset)
df["rating"].value_counts().sort_index()

rating
1     29
2     43
3     61
4    160
5    207
Name: count, dtype: int64

In [5]:
df["rating"].describe()

count    500.000000
mean       3.946000
std        1.184013
min        1.000000
25%        3.000000
50%        4.000000
75%        5.000000
max        5.000000
Name: rating, dtype: float64

---
## 3. Filter to a meaningful subset — what’s in it?

**Meaningful subset:** reviews with **`rating <= 2`** — critical feedback worth investigating separately from neutral or positive reviews.

We store the result in **`subset`** and inspect size plus a sample.

In [6]:
subset = df[df["rating"] <= 2].copy()
print(f"Subset size: {len(subset)} rows (from {len(df)} total)")
subset.head(10)

Subset size: 72 rows (from 500 total)


,id,app,category,rating,review,date,helpful_votes,verified_purchase,device_type,app_version
0,1,Fieldkit,field research,1,Auto-transcription accuracy on accented speake...,2023-03-31,37,True,mobile,2.5.0
1,2,Fieldkit,field research,2,Search results are slow when the repository is...,2024-07-28,12,True,mobile,2.5.3
26,27,Fieldkit,field research,2,No built-in way to generate a structured debri...,2024-04-04,8,True,mobile,2.5.0
31,32,Maze,usability testing,1,Session sharing links occasionally expire befo...,2024-06-27,10,True,tablet,4.2.3
32,33,Lookback,user research,2,Loading large projects takes noticeably longer...,2024-11-22,15,True,desktop,5.2.0
42,43,Fieldkit,field research,2,Search results are slow when the repository is...,2024-07-19,9,True,mobile,NaN
51,52,Maze,usability testing,1,Session sharing links occasionally expire befo...,2024-09-12,32,True,mobile,NaN
74,75,Miro,collaborative whiteboard,1,I've lost tags twice after a session due to a ...,2023-12-25,14,True,desktop,NaN
79,80,Dovetail,research repository,2,The Figma integration is read-only; I can't pu...,2024-08-02,2,True,desktop,1.8.4
102,103,Miro,collaborative whiteboard,2,Storage limits hit quickly when you're recordi...,2024-09-10,16,True,desktop,NaN


**What’s in it:** Only **low-star** reviews (1–2). Use this slice to read themes in dissatisfied feedback or compare **`helpful_votes`** / **`device_type`** completeness vs the full table.

---
## 4. Group by a category and find the average of a numeric column

We **`groupby('category')`** (research tool type) and take **`mean()`** on **`rating`** so we can compare **average star rating** across product categories in this dataset.

In [7]:
df.groupby("category", as_index=False)["rating"].mean().round(3).sort_values("rating")

,category,rating
1,field research,3.674
4,user research,3.905
3,usability testing,4.000
0,collaborative whiteboard,4.017
2,research repository,4.124


---
## 5. What are the missing values? Are any columns incomplete?

**`isnull().sum()`** counts **NaN** per column. Text columns sometimes use **empty strings** instead of NaN; the second cell checks those.

In [8]:
df.isnull().sum()

id                     0
app                    0
category               0
rating                 0
review                 0
date                   0
helpful_votes          0
verified_purchase      0
device_type           63
app_version          111
dtype: int64

In [9]:
# Empty strings in object columns (may not show up in isnull)
for col in df.select_dtypes(include="object").columns:
    blank = (df[col].astype(str).str.strip() == "").sum()
    if blank:
        print(col, "blank or whitespace-only:", blank)

**Readout for this demo file:** Expect **`device_type`** ~63 **NaN** and **`app_version`** ~111 **NaN** (versions not recorded for many rows). Core fields like **`rating`**, **`review`**, **`app`**, and **`date`** are complete. Any remaining blank-like strings in object columns show up in the loop above.

For analysis, you may later **`fillna`**, drop rows, or map missing **`device_type`** to `\"Unknown\"` — see `week5_pandas_demo.ipynb` for patterns.